# Regression track — delivery lead time (dual framing)

**Problem.** For each order approved on the inference date, predict how many calendar days it will take from approval to customer delivery.
- Stakeholder: customer service / fulfilment planning. They get a day-level delay estimate that complements the late/on-time flag from `model_dev_v2.ipynb`.
- Dual framing (Example 4 of the brief): the same underlying outcome drives both tasks.
  - Regression predicts the lead time.
  - Classification predicts whether the lead time exceeds the promised lead time.

**What this notebook covers (work in progress, not final):**
1. Target definition, plus a censoring/horizon audit. This mirrors the v2 buffer audit.
2. Training data at one run date, using the same `get_required_dates` window as v2.
3. A chronological train/test split and day-blocked time-series CV. This replaces the random stratified split.
4. Baselines first:
   - Dummy median.
   - Olist's own promised lead time.
   - The Linear family (LinearRegression → Ridge).
   - The Tree family (DecisionTree → RandomForest → LightGBM).
5. Hyperparameter tuning with CV, holdout metrics, permutation importance and residuals.
6. The late flag derived from the regression output, compared with a classifier.
7. A rolling monthly inference backtest.

Shared code lives in `phase2/src/` and is extracted from `model_dev_v2.ipynb` without changing its behaviour.

# Updates
- 20260927 - Initial regression pipeline, src/ extraction, chronological CV

# Environment set-up

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import RandomizedSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

from src import RANDOM_STATE, LOOKBACK, PXD
from src.data import load_olist_data
from src.features import (build_feature_table, add_extra_features,
                          NUM_COLS, CAT_COLS, EXTRA_NUM_COLS, LEAKAGE_COLS)
from src.labels import (get_required_dates, labels_as_of, regression_targets_as_of,
                        attach_regression_actuals)
from src.splits import chronological_split, day_blocked_time_series_folds
from src.evaluation import regression_metrics, classification_metrics

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
np.random.seed(RANDOM_STATE)

# Constants

In [ ]:
RUN_DATE = '2018-06-02'   # Same run date as model_dev_v2.ipynb
HORIZON = LOOKBACK        # Cap for the lead-time target, chosen in the horizon audit below
TEST_DAYS = 30            # Newest approval days held out as the test set (~1 month)
N_CV_FOLDS = 5
EXCLUDED_STATUSES = ['canceled', 'unavailable']  # Never delivered: not a lead-time outcome

REG_NUM_COLS = NUM_COLS + EXTRA_NUM_COLS
REG_FEATURE_COLS = REG_NUM_COLS + CAT_COLS
assert not set(REG_FEATURE_COLS) & set(LEAKAGE_COLS), 'Outcome columns used as features'

# Load data and build features

In [ ]:
olist = load_olist_data()
orders = olist['orders']
final_df = add_extra_features(build_feature_table(olist), olist)
print(f'Shape: {final_df.shape}; unique orders: {final_df["order_id"].nunique():,}')

# Target definition and horizon audit
**Target:** `lead_days` = customer delivery date − approval date, in calendar days, capped at `HORIZON`:

$$y = \min(\text{lead\_days},\ H)$$

**Why a cap?** At run date *R*, an order approved on day *d* has been observed for only *R − d* days.
- A slow order that is still undelivered has no lead time yet, so it is **right-censored**.
- The v2 classification labels count these orders as late (`late_undelivered`: 1,150 orders at 2018-06-02).
- Dropping them from a regression would remove exactly the worst deliveries and bias predictions downward.

With the cap, a still-undelivered order that is already *H* days old is known to have y = H. Every training order is at least `LOOKBACK` + 1 days old, so with *H* ≤ `LOOKBACK` the capped target is **fully observed for every valid training order**, using only information before the run date.

**Population.** Canceled and unavailable orders never get delivered, so they have no lead time; they are excluded. Orders that were shipped but never delivered are kept and capped.
- Limitation: the dataset only stores each order's final status, so the exclusion uses that snapshot. In training windows (≥ 46 days old) the cancellation would normally already be known.
- At inference time these orders are scored like any other.

In [ ]:
delivered = final_df['order_delivered_customer_date'].notna()
lead_days = (final_df['order_delivered_customer_date'].dt.normalize() - final_df['order_approved_dt']).dt.days
display(lead_days[delivered].describe(percentiles=[.5, .75, .9, .95, .99]).to_frame('lead_days (delivered)'))

audit_rows = []
for run_date in ['2018-02-01', '2018-04-01', '2018-06-01', '2018-08-01']:
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    window = final_df[final_df['order_approved_dt'].between(start, end)
                      & ~final_df['order_status'].isin(EXCLUDED_STATUSES)]
    for horizon in [21, 30, 45]:
        targets = regression_targets_as_of(window, run_date=run_date, horizon=horizon)
        status = targets['target_status'].value_counts(normalize=True) * 100
        audit_rows.append({'run_date': run_date, 'horizon': horizon, 'orders': len(targets),
                           'known_pct': 100 * targets['target_as_of_run'].notna().mean(),
                           'capped_pct': status.get('capped_delivered', 0) + status.get('capped_undelivered', 0),
                           'unresolved_pct': status.get('unresolved', 0)})
horizon_audit = pd.DataFrame(audit_rows)
display(horizon_audit.round(2))

**Reading the audit.**
- At *H* = 45, about 1% of delivered orders exceed the cap. The cap keeps almost all of the lead-time variation and every target is known.
- At *H* = 30, around 4% of orders are capped. That loses detail in exactly the long-delay tail the stakeholder cares about.
- **Decision: `HORIZON = LOOKBACK = 45`.** One buffer then serves both tasks.

# Training data at the run date

In [ ]:
train_test_start_date, train_test_end_date, inference_date = get_required_dates(
    RUN_DATE, lookback=LOOKBACK, pXd=PXD)
window = final_df[final_df['order_approved_dt'].between(train_test_start_date, train_test_end_date)]
excluded = window['order_status'].isin(EXCLUDED_STATUSES)
print(f'Excluded never-delivered orders ({", ".join(EXCLUDED_STATUSES)}): {excluded.sum():,}')

reg_audit = regression_targets_as_of(window[~excluded], run_date=RUN_DATE, horizon=HORIZON)
display(reg_audit['target_status'].value_counts().to_frame('orders'))
reg_df = reg_audit[reg_audit['target_as_of_run'].notna()].copy()
reg_df['y'] = reg_df['target_as_of_run'].astype(float)
print(f'Training/test orders with a known target: {len(reg_df):,}')

# The classification label for the same orders, for the dual-framing comparison later
reg_df['late_label'] = labels_as_of(reg_df, run_date=RUN_DATE)['label_as_of_run']

# Chronological split
- **Test:** the last `TEST_DAYS` approval days of the window, about one month of orders, like v2.
- **Train:** everything before that.

The v2 notebook uses `train_test_split(..., stratify=y)`, which scatters later orders into training. The section *Why not a random split?* measures how optimistic that makes the scores.

In [ ]:
train_df, test_df = chronological_split(reg_df, date_col='order_approved_dt', test_days=TEST_DAYS)
train_df = train_df.sort_values('order_approved_dt', kind='stable')
X_train, y_train = train_df[REG_FEATURE_COLS], train_df['y']
X_test, y_test = test_df[REG_FEATURE_COLS], test_df['y']
print(f'Train: {train_df["order_approved_dt"].min():%Y-%m-%d} to {train_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(train_df):,} orders, mean y {y_train.mean():.2f})')
print(f'Test:  {test_df["order_approved_dt"].min():%Y-%m-%d} to {test_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(test_df):,} orders, mean y {y_test.mean():.2f})')

# Expanding-window folds on the training part; no approval day is split across folds
cv_folds = day_blocked_time_series_folds(train_df['order_approved_dt'], n_splits=N_CV_FOLDS)
for k, (tr, va) in enumerate(cv_folds, 1):
    print(f'Fold {k}: train {len(tr):>6,} orders -> validate {len(va):>5,} orders '
          f'({train_df["order_approved_dt"].iloc[va].min():%Y-%m-%d} to {train_df["order_approved_dt"].iloc[va].max():%Y-%m-%d})')

# Models
Two families across the whole project, as the brief's model-family budget requires. Each starts from its simplest variant:

| Family | Baseline | More complex |
|---|---|---|
| Linear | LinearRegression | Ridge (tuned α) |
| Tree-based | DecisionTreeRegressor (default) | RandomForestRegressor, LightGBM (tuned) |

There are also two reference baselines:
- `DummyRegressor(median)`.
- **Olist promise**: predict the promised lead time, capped at *H*. This is the bar that matters to the business: *does the model estimate delivery better than the date Olist already shows the customer?*

All preprocessing is fitted inside each pipeline:
- Median imputation, with scaling for the linear models only.
- One-hot encoding of categorical columns.

In [ ]:
def make_preprocessor(scale):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), REG_NUM_COLS),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ], sparse_threshold=0)

def make_pipeline(model, scale=False):
    return Pipeline([('preprocessor', make_preprocessor(scale)), ('model', model)])

class PromiseBaseline(DummyRegressor):
    """Predicts Olist's promised lead time (capped at HORIZON); ignores training targets."""
    def fit(self, X, y, sample_weight=None):
        return super().fit(X, y)
    def predict(self, X):
        return np.clip(X['promised_lead_days'].to_numpy(dtype=float), 0, HORIZON)

baseline_models = {
    'Dummy (median)': DummyRegressor(strategy='median'),
    'Olist promise': PromiseBaseline(),
    'Linear: LinearRegression': make_pipeline(LinearRegression(), scale=True),
    'Tree: DecisionTree (default)': make_pipeline(DecisionTreeRegressor(random_state=RANDOM_STATE)),
}

## Cross-validated baselines

In [ ]:
SCORING = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}

def cv_summary(name, model):
    scores = cross_validate(model, X_train, y_train, cv=cv_folds, scoring=SCORING, n_jobs=-1)
    row = {'model': name}
    for metric in SCORING:
        values = scores[f'test_{metric}'] * (1 if metric == 'r2' else -1)
        row[f'cv_{metric}_mean'], row[f'cv_{metric}_std'] = values.mean(), values.std()
    return row

cv_rows = [cv_summary(name, model) for name, model in baseline_models.items()]
display(pd.DataFrame(cv_rows).set_index('model'))

## Hyperparameter tuning (RandomizedSearchCV on the time-series folds)
Each search uses the same day-blocked folds and optimises MAE, which reads directly as "days off". The grids are small on purpose; widen them once the feature set is frozen.

In [ ]:
search_spaces = {
    'Linear: Ridge': (make_pipeline(Ridge(), scale=True),
                      {'model__alpha': np.logspace(-2, 3, 12)}),
    'Tree: RandomForest': (make_pipeline(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
                           {'model__n_estimators': [100, 200],
                            'model__max_depth': [8, 12, 16],
                            'model__min_samples_leaf': [10, 30, 60],
                            'model__max_features': [0.3, 0.5]}),
    'Tree: LightGBM': (make_pipeline(LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1, n_jobs=1)),
                       {'model__n_estimators': [200, 400, 600],
                        'model__learning_rate': [0.02, 0.05, 0.1],
                        'model__num_leaves': [15, 31, 63],
                        'model__min_child_samples': [20, 50, 100],
                        'model__subsample': [0.7, 1.0], 'model__subsample_freq': [1],
                        'model__colsample_bytree': [0.6, 0.8, 1.0],
                        'model__objective': ['regression', 'regression_l1']}),
}
# Kept small for laptop run times. Models use n_jobs=1 so parallelism happens only
# across CV folds, which avoids oversubscribing CPU cores.
N_ITER = {'Linear: Ridge': 12, 'Tree: RandomForest': 6, 'Tree: LightGBM': 10}

tuned_models, best_params = {}, {}
for name, (pipeline, space) in search_spaces.items():
    search = RandomizedSearchCV(pipeline, space, n_iter=N_ITER[name], cv=cv_folds,
                                scoring='neg_mean_absolute_error', random_state=RANDOM_STATE,
                                n_jobs=-1, refit=False)
    search.fit(X_train, y_train)
    best_params[name] = search.best_params_
    tuned_models[name] = pipeline.set_params(**search.best_params_)
    cv_rows.append(cv_summary(name, tuned_models[name]))
    print(f'{name}: best CV MAE {-search.best_score_:.3f} with {search.best_params_}')

cv_results = pd.DataFrame(cv_rows).set_index('model')
display(cv_results)

# Holdout evaluation (last 30 days of the window)

In [ ]:
all_models = {**baseline_models, **tuned_models}
holdout_rows, test_predictions = [], {}
for name, model in all_models.items():
    model.fit(X_train, y_train)
    prediction = np.clip(model.predict(X_test), 0, HORIZON)
    test_predictions[name] = prediction
    holdout_rows.append({'model': name, **regression_metrics(y_test, prediction)})
holdout_results = pd.DataFrame(holdout_rows).set_index('model').sort_values('mae')
display(holdout_results)

best_name = cv_results.drop(index=['Dummy (median)', 'Olist promise'])['cv_mae_mean'].idxmin()
best_model = all_models[best_name]
print(f'Selected by CV MAE (not by test score): {best_name}')
for reference in ['Dummy (median)', 'Olist promise']:
    gain = holdout_results.loc[reference, 'mae'] - holdout_results.loc[best_name, 'mae']
    print(f'  Test MAE improvement over {reference}: {gain:.2f} days')

## Why not a random split?
This refits the selected model on a random 1/6 split of the same window, the way v2 splits, and compares its scores with the chronological split above. If the random split scores better, that difference is optimism from mixing future orders into training.

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    reg_df[REG_FEATURE_COLS], reg_df['y'], test_size=TEST_DAYS / PXD, random_state=RANDOM_STATE)
random_split_model = clone(all_models[best_name])
random_split_model.fit(Xr_train, yr_train)
display(pd.DataFrame({
    'random split (v2 style)': regression_metrics(yr_test, np.clip(random_split_model.predict(Xr_test), 0, HORIZON)),
    'chronological split': holdout_results.loc[best_name],
}).T)

# Interpretation

## Permutation importance (test set, raw features)

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
importance = (pd.DataFrame({'feature': REG_FEATURE_COLS,
                            'mae_increase_days': perm.importances_mean,
                            'std': perm.importances_std})
              .sort_values('mae_increase_days', ascending=False).head(20).reset_index(drop=True))
display(importance)

## Residuals by segment

In [ ]:
residuals = test_df[['customer_state', 'route_type']].copy()
residuals['actual'] = y_test.to_numpy()
residuals['predicted'] = test_predictions[best_name]
residuals['error'] = residuals['predicted'] - residuals['actual']
for segment in ['route_type', 'customer_state']:
    table = residuals.groupby(segment).agg(orders=('error', 'size'), actual_mean=('actual', 'mean'),
                                           predicted_mean=('predicted', 'mean'), bias=('error', 'mean'),
                                           mae=('error', lambda e: e.abs().mean()))
    display(table.sort_values('orders', ascending=False).head(12))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(residuals['actual'], residuals['predicted'], s=4, alpha=0.3)
axes[0].plot([0, HORIZON], [0, HORIZON], color='red', linestyle=':')
axes[0].set(xlabel='Actual capped lead days', ylabel='Predicted', title=f'{best_name}: predicted vs actual (test)')
axes[1].hist(residuals['error'], bins=60)
axes[1].set(xlabel='Predicted - actual (days)', ylabel='Orders', title='Residual distribution')
plt.tight_layout()
plt.show()

# Dual framing: a late flag derived from the regression output
An order is flagged late when its **predicted lead time exceeds the promised lead time**. This is the same outcome as v2's classification label.

The table compares this flag on the same chronological test set with:
- a LightGBM classifier trained on the same features and split, at a 0.5 threshold;
- a Logistic Regression classifier.

It also checks what the new features add to the classifier. PR-AUC is compared with v2's base feature set (`NUM_COLS + CAT_COLS`) and with the extended one.

In [ ]:
test_late = test_df['late_label'].astype(int)
train_late = train_df['late_label'].astype(int)
margin = test_predictions[best_name] - np.clip(test_df['promised_lead_days'].to_numpy(dtype=float), 0, HORIZON)
dual_rows = {'Regression-derived flag': classification_metrics(test_late, (margin > 0).astype(int), margin)}

def fit_classifier(model, num_cols, scale):
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]
                             + ([('scaler', StandardScaler())] if scale else [])), num_cols),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)], sparse_threshold=0)
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(train_df[num_cols + CAT_COLS], train_late)
    prob = pipeline.predict_proba(test_df[num_cols + CAT_COLS])[:, 1]
    return classification_metrics(test_late, (prob >= 0.5).astype(int), prob)

ratio = (train_late == 0).sum() / (train_late == 1).sum()
for label, cols in [('v2 features', NUM_COLS), ('v2 + extra features', REG_NUM_COLS)]:
    dual_rows[f'LightGBM classifier ({label})'] = fit_classifier(
        LGBMClassifier(scale_pos_weight=ratio, random_state=RANDOM_STATE, verbosity=-1), cols, scale=False)
    dual_rows[f'LogisticRegression ({label})'] = fit_classifier(
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE), cols, scale=True)
print(f'Test late rate: {test_late.mean():.2%}')
display(pd.DataFrame(dual_rows).T)

# Rolling monthly inference backtest
This repeats v2's operational flow for several months:
1. Train on the 2nd of each month.
2. Score every order approved in that month on its approval day.
3. Evaluate each daily cohort once its capped target is known, at approval day + `HORIZON`.

The delivery data ends in mid-October 2018, so the last month that can be fully evaluated is July 2018.

In [ ]:
def monthly_backtest(model_template, run_date):
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    history = final_df[final_df['order_approved_dt'].between(start, end)
                       & ~final_df['order_status'].isin(EXCLUDED_STATUSES)]
    targets = regression_targets_as_of(history, run_date=run_date, horizon=HORIZON)
    targets = targets[targets['target_as_of_run'].notna()]
    model = clone(model_template)
    model.fit(targets[REG_FEATURE_COLS], targets['target_as_of_run'].astype(float))

    month = pd.Period(run_date, 'M')
    cohort = final_df[final_df['order_approved_dt'].dt.to_period('M').eq(month)]
    predictions = cohort[['order_id', 'order_approved_dt', 'promised_lead_days']].copy()
    predictions['predicted'] = np.clip(model.predict(cohort[REG_FEATURE_COLS]), 0, HORIZON)
    predictions = attach_regression_actuals(predictions, orders, horizon=HORIZON)
    known = predictions['actual_target'].notna()
    promise = np.clip(predictions.loc[known, 'promised_lead_days'].astype(float), 0, HORIZON)
    actual = predictions.loc[known, 'actual_target'].astype(float)
    return {'month': str(month), 'train_orders': len(targets), 'scored_orders': len(predictions),
            'known_pct': 100 * known.mean(),
            'mae': regression_metrics(actual, predictions.loc[known, 'predicted'])['mae'],
            'mae_olist_promise': regression_metrics(actual, promise)['mae'],
            'rmse': regression_metrics(actual, predictions.loc[known, 'predicted'])['rmse'],
            'r2': regression_metrics(actual, predictions.loc[known, 'predicted'])['r2']}

BACKTEST_RUN_DATES = [f'2018-{m:02d}-02' for m in range(1, 8)]
backtest = pd.DataFrame([monthly_backtest(best_model, d) for d in BACKTEST_RUN_DATES]).set_index('month')
display(backtest)
print(f"MAE across months: {backtest['mae'].mean():.2f} ± {backtest['mae'].std():.2f} days "
      f"(Olist promise: {backtest['mae_olist_promise'].mean():.2f})")

# Next steps (not done yet)
- **Seller/state history features, computed as of the run date.** Examples: seller median lead time and seller late rate over prior labelled orders. Build them from `regression_targets_as_of` / `labels_as_of` using a history window that ends before each training row's own label is known. Otherwise they leak.
- **Threshold selection for the classifier.** Choose the threshold on the CV folds (max F1, or a capacity-based top-k decile) instead of 0.5.
- **Optional stacking.** Build a Ridge + LightGBM `StackingRegressor`. Keep it only if its CV MAE beats the best single model by more than one CV standard deviation.
- **Team decisions:**
  - Adopt the chronological split and `src/` helpers in `model_dev_v2.ipynb`.
  - Prune the classifiers to the same two families.
  - Confirm how canceled/unavailable orders are handled.
- **Report limitations:**
  - The capped target hides detail beyond 45 days (~1% of orders).
  - Estimated dates are assumed fixed.
  - The dataset holds only final order statuses.
  - The May 2018 truckers' strike month.